# Generate Simulated Dynamic Homogeneous Link Data

This notebook creates larger and more learnable snapshot data for `DyMLH-Link`.

The setting is:

```text
2015, 2016, 2017, 2018, 2019 snapshots -> predict 2020 links
```

The historical snapshots contain node features and `global_id`. The target graph `graph_2020.bin` stores positive target edges with `train_mask`, `valid_mask`, and `test_mask`.

Important: negative samples are not saved in the `.bin` files. They are sampled dynamically during training. The current training loader excludes all positive edges from the target year and all historical snapshots, so a sampled negative will not be a pair that was connected in 2015-2020.


In [ ]:
from pathlib import Path

import numpy as np
import torch
import dgl

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

OUT_DIR = Path('data/simulated')
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2015, 2021))
HISTORY_YEARS = YEARS[:-1]
TARGET_YEAR = YEARS[-1]

NUM_GLOBAL_NODES = 3000
NUM_COMMUNITIES = 8
FEAT_DIM = 64
LATENT_DIM = 32

BASE_ACTIVE_PROB = 0.88
NEW_NODE_DRIFT = 0.015
STABLE_CORE_RATIO = 0.55

MAX_RANDOM_PAIRS = 1_200_000
PREVIOUS_EDGE_KEEP_PROB = 0.72
EDGE_BIAS = -2.1
SAME_COMMUNITY_BONUS = 2.3
PREVIOUS_EDGE_BONUS = 2.6
RECENT_EDGE_BONUS = 1.0
MAKE_UNDIRECTED = True

TRAIN_RATIO = 0.7
VALID_RATIO = 0.1
TEST_RATIO = 0.2

print('Output directory:', OUT_DIR.resolve())


## Simulation Design

The data is generated from three signals:

1. Community membership: nodes in the same community are more likely to connect.
2. Evolving latent vectors: node features and edges share an underlying latent space.
3. Link persistence: edges from previous years are more likely to reappear in later years.

This makes the prediction task learnable: historical structure and node features both help predict 2020 links.


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


community = rng.integers(0, NUM_COMMUNITIES, size=NUM_GLOBAL_NODES, dtype=np.int64)
community_centers = rng.normal(0, 1.2, size=(NUM_COMMUNITIES, LATENT_DIM)).astype(np.float32)
node_offsets = rng.normal(0, 0.55, size=(NUM_GLOBAL_NODES, LATENT_DIM)).astype(np.float32)
feature_projection = rng.normal(0, 0.45, size=(LATENT_DIM + NUM_COMMUNITIES, FEAT_DIM)).astype(np.float32)


def active_nodes_for_year(year_idx):
    active_prob = min(0.98, BASE_ACTIVE_PROB + NEW_NODE_DRIFT * year_idx)
    active = rng.random(NUM_GLOBAL_NODES) < active_prob
    active[: int(NUM_GLOBAL_NODES * STABLE_CORE_RATIO)] = True
    return np.flatnonzero(active).astype(np.int64)


def latent_for_year(global_ids, year_idx):
    slow_drift = 0.06 * year_idx * rng.normal(0, 1, size=(NUM_COMMUNITIES, LATENT_DIM)).astype(np.float32)
    centers = community_centers[community[global_ids]] + slow_drift[community[global_ids]]
    year_noise = rng.normal(0, 0.12, size=(len(global_ids), LATENT_DIM)).astype(np.float32)
    return centers + node_offsets[global_ids] + year_noise


def features_from_latent(global_ids, latent):
    one_hot = np.eye(NUM_COMMUNITIES, dtype=np.float32)[community[global_ids]]
    raw = np.concatenate([latent, one_hot], axis=1)
    noise = rng.normal(0, 0.03, size=(len(global_ids), FEAT_DIM)).astype(np.float32)
    return raw @ feature_projection + noise


def unique_pairs(src, dst):
    keep = src != dst
    src, dst = src[keep], dst[keep]
    if len(src) == 0:
        return src.astype(np.int64), dst.astype(np.int64)
    pairs = np.unique(np.stack([src, dst], axis=1), axis=0)
    return pairs[:, 0].astype(np.int64), pairs[:, 1].astype(np.int64)


def localize_previous_edges(previous_edges, global_to_local):
    src, dst = [], []
    for u, v in previous_edges:
        if u in global_to_local and v in global_to_local:
            src.append(global_to_local[u])
            dst.append(global_to_local[v])
    if not src:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64)
    return np.asarray(src, dtype=np.int64), np.asarray(dst, dtype=np.int64)


def sample_edges(global_ids, latent, previous_edges, recent_edges):
    n = len(global_ids)
    if n < 2:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64), set()

    global_to_local = {int(gid): idx for idx, gid in enumerate(global_ids.tolist())}

    rand_src = rng.integers(0, n, size=MAX_RANDOM_PAIRS, dtype=np.int64)
    rand_dst = rng.integers(0, n, size=MAX_RANDOM_PAIRS, dtype=np.int64)

    prev_src, prev_dst = localize_previous_edges(previous_edges, global_to_local)
    if len(prev_src) > 0:
        keep_prev = rng.random(len(prev_src)) < PREVIOUS_EDGE_KEEP_PROB
        prev_src, prev_dst = prev_src[keep_prev], prev_dst[keep_prev]

    src = np.concatenate([rand_src, prev_src])
    dst = np.concatenate([rand_dst, prev_dst])
    src, dst = unique_pairs(src, dst)

    src_gid = global_ids[src]
    dst_gid = global_ids[dst]
    same_comm = (community[src_gid] == community[dst_gid]).astype(np.float32)
    latent_score = np.sum(latent[src] * latent[dst], axis=1) / np.sqrt(latent.shape[1])
    prev_flag = np.array([(int(u), int(v)) in previous_edges for u, v in zip(src_gid, dst_gid)], dtype=np.float32)
    recent_flag = np.array([(int(u), int(v)) in recent_edges for u, v in zip(src_gid, dst_gid)], dtype=np.float32)

    score = EDGE_BIAS + latent_score + SAME_COMMUNITY_BONUS * same_comm + PREVIOUS_EDGE_BONUS * prev_flag + RECENT_EDGE_BONUS * recent_flag
    prob = sigmoid(score)
    chosen = rng.random(len(prob)) < prob
    src, dst = src[chosen], dst[chosen]
    src, dst = unique_pairs(src, dst)

    if MAKE_UNDIRECTED and len(src) > 0:
        src, dst = unique_pairs(np.concatenate([src, dst]), np.concatenate([dst, src]))

    edge_set = set(zip(global_ids[src].tolist(), global_ids[dst].tolist()))
    return src, dst, edge_set


def make_masks(num_edges):
    order = rng.permutation(num_edges)
    n_train = int(num_edges * TRAIN_RATIO)
    n_valid = int(num_edges * VALID_RATIO)
    train_idx = order[:n_train]
    valid_idx = order[n_train:n_train + n_valid]
    test_idx = order[n_train + n_valid:]

    train_mask = torch.zeros(num_edges, dtype=torch.bool)
    valid_mask = torch.zeros(num_edges, dtype=torch.bool)
    test_mask = torch.zeros(num_edges, dtype=torch.bool)
    train_mask[torch.as_tensor(train_idx, dtype=torch.long)] = True
    valid_mask[torch.as_tensor(valid_idx, dtype=torch.long)] = True
    test_mask[torch.as_tensor(test_idx, dtype=torch.long)] = True
    return train_mask, valid_mask, test_mask


## Generate and Save Graphs

The target year graph stores split masks on positive edges. Historical years do not store edge masks.


In [ ]:
saved_paths = []
previous_edges = set()
recent_edges = set()
year_edge_sets = {}

for year_idx, year in enumerate(YEARS):
    global_ids = active_nodes_for_year(year_idx)
    latent = latent_for_year(global_ids, year_idx)
    features = features_from_latent(global_ids, latent)
    src, dst, edge_set = sample_edges(global_ids, latent, previous_edges, recent_edges)

    graph = dgl.graph((torch.from_numpy(src), torch.from_numpy(dst)), num_nodes=len(global_ids))
    graph.ndata['feat'] = torch.tensor(features, dtype=torch.float32)
    graph.ndata['global_id'] = torch.tensor(global_ids, dtype=torch.long)
    graph.ndata['community'] = torch.tensor(community[global_ids], dtype=torch.long)

    if year == TARGET_YEAR:
        train_mask, valid_mask, test_mask = make_masks(graph.num_edges())
        graph.edata['train_mask'] = train_mask
        graph.edata['valid_mask'] = valid_mask
        graph.edata['test_mask'] = test_mask

    path = OUT_DIR / f'graph_{year}.bin'
    dgl.save_graphs(str(path), [graph], {'year': torch.tensor([year])})
    saved_paths.append(path)
    year_edge_sets[year] = edge_set

    print(year, graph, 'saved to', path)
    print('  active nodes:', len(global_ids), 'edges:', graph.num_edges())

    recent_edges = edge_set
    previous_edges |= edge_set


## Inspect Temporal Signal

This cell reports how much the target year overlaps with historical edges. Some overlap is intentional: it creates a dynamic link-persistence pattern that temporal models can learn.


In [ ]:
target_edges = year_edge_sets[TARGET_YEAR]
history_edges = set().union(*(year_edge_sets[y] for y in HISTORY_YEARS))
overlap = len(target_edges & history_edges)
print('target edges:', len(target_edges))
print('historical unique edges:', len(history_edges))
print('target edges previously observed:', overlap)
print('overlap ratio:', overlap / max(1, len(target_edges)))

target_graph, metadata = dgl.load_graphs(str(OUT_DIR / f'graph_{TARGET_YEAR}.bin'))
g = target_graph[0]
print(g)
print('metadata:', metadata)
print('node data:', list(g.ndata.keys()))
print('edge data:', list(g.edata.keys()))
print('train/valid/test edges:', int(g.edata['train_mask'].sum()), int(g.edata['valid_mask'].sum()), int(g.edata['test_mask'].sum()))


## Training Command

After running the cells above, use this command from the repository root. `--undirected` should be kept if `MAKE_UNDIRECTED=True`.


In [ ]:
snapshot_bins = ','.join(str(OUT_DIR / f'graph_{year}.bin') for year in HISTORY_YEARS)
target_bin = str(OUT_DIR / f'graph_{TARGET_YEAR}.bin')
undirected_flag = " \
  --undirected" if MAKE_UNDIRECTED else ""

cmd = f'''python -u Link_Prediction.py \
  --snapshot-bins {snapshot_bins} \
  --target-bin {target_bin} \
  --feat-key feat \
  --global-id-key global_id \
  --hidden-dim 128 \
  --gnn-layers 2 \
  --sage-aggregator-type mean \
  --temporal-model gru \
  --temporal-layers 1 \
  --predictor dot \
  --negative-ratio 1.0 \
  --eval-negative-ratio 1.0 \
  --epochs 200 \
  --patience 30 \
  --early-stop-metric auc \
  --log-every 10 \
  --output-dir outputs{undirected_flag}'''
print(cmd)
